<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">

# **Analisis de Series de Tiempo II**
# **Tarea 1, Feature Engineering y Validacion Temporal con GBM**

## Enunciado completo


**Limite de entrega para ser evaluado sobre 100 puntos:** 15 de julio, 23:59 (UTC-3)

Posteriormente pueden enviar la solucion de la tarea, pero cada semana de atraso reduce 10 puntos la evaluacion. Es decir, del 16 de julio 00:00 al 23 de julio 23:59 (UTC-3) se evaluara sobre 90 puntos, y asi sucesivamente hasta el final de la materia.


La tarea se realiza utilizando la serie mensual AirPassengers (pasajeros de aerolineas).

**Parte 1) Feature engineering, 20 puntos**

A partir de la serie "y", deben construir:
* Features de calendario: mes, trimestre, anio, dia_anio, y sus versiones ciclicas (sin/cos) para mes y dia_anio.
* Features con lags [1, 2, 3, 12].
* Features de ventana movil (rolling) de tamano 3 y 12: media, desvio, maximo y minimo, todas con .shift(1) para no filtrar el valor actual.
* Features expansivas (expanding): media y desvio acumulados hasta el periodo anterior.

**Parte 2) Dos conjuntos de features, 5 puntos**

Deben armar dos grupos:
* features_fecha: solo variables de calendario.
* features_all: calendario + lags + rolling + expanding.

**Parte 3) Dos formas de particionar train/test, 5 puntos**

* Holdout temporal: los ultimos 24 meses como test, todo lo anterior como train (respetando el orden cronologico).
* Holdout con shuffle: mismo tamano de test, pero mezclando filas al azar (train_test_split(shuffle=True)).

**Parte 4) Entrenamiento de Gradient Boosting Regressor, 20 puntos**

Deben entrenar un GradientBoostingRegressor para cada combinacion (2 conjuntos de features x 2 tipos de holdout = 4 modelos).

*Usar GradientBoostingRegressor para este ejercicio, no HistGradientBoostingRegressor. Este ultimo esta optimizado para datasets grandes.*

**Parte 5) Metricas, 10 puntos**

Para cada uno de los 4 modelos, calcular MAE, RMSE, MAPE y MASE (usando como referencia el naive estacional de lag 12, es decir, comparar contra "el mismo mes del anio anterior").

**Parte 6) Interpretabilidad, 20 puntos**

Usando el modelo entrenado con features_all en el holdout temporal (el valido, no el de shuffle):

* Calcular la importancia nativa del modelo (feature_importances_) y graficar el top 10.
* Calcular permutation importance sobre el conjunto de test, con scoring="neg_mean_absolute_error" y al menos 30 repeticiones (n_repeats=30), y grafiquen el top 10.
* Comparar ambos rankings.

**Parte 7) Preguntas de analisis, 20 puntos**

* Cual de los dos holdouts refleja mejor el escenario real de uso del modelo (predecir el futuro)? Argumentar en terminos de "el cuando" vs "el que paso".
* En el holdout temporal, comparar features_fecha vs features_all. Cuanto aporta realmente tener lags/rolling frente a usar solo variables de calendario?
* Interpretar el valor de MASE en el holdout temporal: el modelo le gana o le pierde al naive estacional? Que les dice esto sobre la dificultad de mejorar un baseline fuerte en series cortas y muy estacionales?
* Que feature domina el ranking en ambas tecnicas? Por que tiene sentido que sea esa, dada la naturaleza estacional de la serie?

**Entregable: notebook con el codigo corriendo y las respuestas a las 4 preguntas.**

**Link de entrega: [https://forms.gle/AJLpFTwypFJvjYETA](https://forms.gle/AJLpFTwypFJvjYETA)**


## Resolucion

Alumno: Gustavo Julian Rivas N° SIU: a1626.

La resolucion se presenta siguiendo cada parte solicitada en el enunciado. Se usa una redaccion impersonal para describir el procedimiento, las decisiones tomadas y la interpretacion de los resultados.


## Parte 1) Feature engineering, 20 puntos

Primero se prepara el entorno, se carga la serie AirPassengers y se construyen las variables solicitadas. El notebook esta pensado para ejecutarse de arriba hacia abajo en Google Colab y no requiere GPU.


In [ ]:
# Si alguna dependencia no estuviera disponible en Colab, descomentar la linea siguiente.
# !pip install -q pandas numpy matplotlib scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 24
SEASONAL_PERIOD = 12

plt.style.use("seaborn-v0_8-whitegrid")

### Carga de datos

La consigna propone usar `statsmodels.datasets.get_rdataset("AirPassengers")`. Para que el notebook sea reproducible en Colab aun sin acceso a esa descarga, la serie queda embebida. La serie tiene 144 observaciones mensuales desde enero de 1949 hasta diciembre de 1960.


In [ ]:
AIR_PASSENGERS = [
    112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118,
    115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140,
    145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166,
    171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194,
    196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201,
    204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229,
    242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278,
    284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306,
    315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336,
    340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337,
    360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405,
    417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432,
]

df = pd.DataFrame({
    "ds": pd.date_range("1949-01-01", periods=len(AIR_PASSENGERS), freq="MS"),
    "y": AIR_PASSENGERS,
}).set_index("ds")

display(df.head())
display(df.tail())
print(f"Observaciones: {len(df)} | Desde {df.index.min().date()} hasta {df.index.max().date()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
df["y"].plot(ax=ax, marker="o", linewidth=1.5)
ax.set_title("Serie mensual AirPassengers")
ax.set_xlabel("Fecha")
ax.set_ylabel("Pasajeros")
plt.show()

### Construccion de features

Se crean features de calendario, lags, ventanas moviles y ventanas expansivas. Las features rolling y expanding se calculan con `.shift(1)`, por lo que para predecir `y_t` se usa informacion disponible hasta `t-1`, sin usar el valor actual ni valores futuros.


In [ ]:
def add_calendar_features(data):
    out = data.copy()
    idx = out.index

    out["mes"] = idx.month
    out["trimestre"] = idx.quarter
    out["anio"] = idx.year
    out["dia_anio"] = idx.dayofyear

    out["mes_sin"] = np.sin(2 * np.pi * out["mes"] / 12)
    out["mes_cos"] = np.cos(2 * np.pi * out["mes"] / 12)
    out["dia_anio_sin"] = np.sin(2 * np.pi * out["dia_anio"] / 365.25)
    out["dia_anio_cos"] = np.cos(2 * np.pi * out["dia_anio"] / 365.25)

    return out


def add_lag_features(data, lags=(1, 2, 3, 12)):
    out = data.copy()
    for lag in lags:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    return out


def add_rolling_features(data, windows=(3, 12)):
    out = data.copy()
    for window in windows:
        rolling = out["y"].rolling(window=window)
        out[f"roll_mean_{window}"] = rolling.mean().shift(1)
        out[f"roll_std_{window}"] = rolling.std().shift(1)
        out[f"roll_max_{window}"] = rolling.max().shift(1)
        out[f"roll_min_{window}"] = rolling.min().shift(1)
    return out


def add_expanding_features(data):
    out = data.copy()
    out["exp_mean"] = out["y"].expanding().mean().shift(1)
    out["exp_std"] = out["y"].expanding().std().shift(1)
    return out


df_features = (
    df.pipe(add_calendar_features)
      .pipe(add_lag_features)
      .pipe(add_rolling_features)
      .pipe(add_expanding_features)
)

features_fecha = [
    "mes", "trimestre", "anio", "dia_anio",
    "mes_sin", "mes_cos", "dia_anio_sin", "dia_anio_cos",
]

features_lags = ["lag_1", "lag_2", "lag_3", "lag_12"]
features_rolling = [
    f"roll_{stat}_{window}"
    for window in (3, 12)
    for stat in ("mean", "std", "max", "min")
]
features_expanding = ["exp_mean", "exp_std"]
features_all = features_fecha + features_lags + features_rolling + features_expanding

# Se usa una base comun sin NaN para que ambas familias de features se comparen sobre las mismas fechas.
df_model = df_features.dropna(subset=features_all + ["y"]).copy()

print(f"Primera fecha modelable: {df_model.index.min().date()}")
print(f"Ultima fecha modelable:  {df_model.index.max().date()}")
print(f"Filas modelables: {len(df_model)}")
print(f"Cantidad features_fecha: {len(features_fecha)}")
print(f"Cantidad features_all:   {len(features_all)}")

display(df_model.head())

## Parte 2) Dos conjuntos de features, 5 puntos

Se arman los dos grupos pedidos por la consigna. `features_fecha` contiene solo variables de calendario. `features_all` combina calendario, lags, rolling windows y expanding windows. La celda anterior deja definidos ambos grupos y muestra la cantidad de variables de cada conjunto.


## Parte 3) Dos formas de particionar train/test, 5 puntos

Se comparan dos particiones. En el holdout temporal, los ultimos 24 meses se reservan como test y todo lo anterior se usa como train. En el holdout con shuffle, se mantiene el mismo tamano de test, pero las filas se mezclan al azar mediante `train_test_split(shuffle=True)`.


In [ ]:
split_position = len(df_model) - TEST_SIZE
train_temporal = df_model.iloc[:split_position].copy()
test_temporal = df_model.iloc[split_position:].copy()

print("Holdout temporal")
print(f"Train: {train_temporal.index.min().date()} a {train_temporal.index.max().date()} | n={len(train_temporal)}")
print(f"Test:  {test_temporal.index.min().date()} a {test_temporal.index.max().date()} | n={len(test_temporal)}")

fig, ax = plt.subplots(figsize=(12, 4))
train_temporal["y"].plot(ax=ax, label="Train temporal")
test_temporal["y"].plot(ax=ax, label="Test temporal", color="tab:orange")
ax.axvline(test_temporal.index.min(), color="black", linestyle="--", alpha=0.7)
ax.set_title("Holdout temporal: pasado para entrenar, futuro para testear")
ax.set_ylabel("Pasajeros")
ax.legend()
plt.show()

## Parte 4) Entrenamiento de Gradient Boosting Regressor, 20 puntos

Se entrena `GradientBoostingRegressor` para cada combinacion entre conjunto de features y tipo de holdout. Antes del entrenamiento se definen las funciones de metricas, porque la evaluacion queda integrada en la funcion de entrenamiento.


In [ ]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


def mase_scale_from_train(train_y, m=12):
    train_y = pd.Series(train_y).reset_index(drop=True)
    return np.mean(np.abs(train_y.iloc[m:].to_numpy() - train_y.iloc[:-m].to_numpy()))


def evaluate_model(y_true, y_pred, scale):
    model_mae = mae(y_true, y_pred)
    return {
        "MAE": model_mae,
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "MASE": model_mae / scale,
    }


mase_scale = mase_scale_from_train(train_temporal["y"], m=SEASONAL_PERIOD)
print(f"Escala MASE naive estacional lag {SEASONAL_PERIOD}: {mase_scale:.3f}")

In [ ]:
def make_gbm():
    return GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        random_state=RANDOM_STATE,
    )


def fit_and_eval(feature_names, holdout_name):
    if holdout_name == "temporal":
        train_data = train_temporal
        test_data = test_temporal
        X_train = train_data[feature_names]
        X_test = test_data[feature_names]
        y_train = train_data["y"]
        y_test = test_data["y"]
    elif holdout_name == "shuffle":
        X_train, X_test, y_train, y_test = train_test_split(
            df_model[feature_names],
            df_model["y"],
            test_size=TEST_SIZE,
            shuffle=True,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError("holdout_name debe ser 'temporal' o 'shuffle'")

    model = make_gbm()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = evaluate_model(y_test, pred, mase_scale)

    return {
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "pred": pd.Series(pred, index=y_test.index, name="pred"),
        "metrics": metrics,
    }

experiments = {}
rows = []

feature_sets = {
    "features_fecha": features_fecha,
    "features_all": features_all,
}

for feature_set_name, feature_names in feature_sets.items():
    for holdout_name in ["temporal", "shuffle"]:
        result = fit_and_eval(feature_names, holdout_name)
        experiments[(feature_set_name, holdout_name)] = result
        rows.append({
            "features": feature_set_name,
            "holdout": holdout_name,
            **result["metrics"],
        })

results = (
    pd.DataFrame(rows)
      .sort_values(["holdout", "MASE"])
      .reset_index(drop=True)
)

display(results.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "MAPE": "{:.2f}", "MASE": "{:.3f}"}))

## Parte 5) Metricas, 10 puntos

Para cada uno de los cuatro modelos se calculan MAE, RMSE, MAPE y MASE. MASE se calcula con una escala fija obtenida desde el naive estacional de lag 12 sobre el entrenamiento temporal, de modo que todos los modelos se comparan contra la misma referencia.


In [ ]:
display(results.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "MAPE": "{:.2f}", "MASE": "{:.3f}"}))


In [ ]:
temporal_plot = pd.DataFrame({
    "real": test_temporal["y"],
    "GBM features_fecha": experiments[("features_fecha", "temporal")]["pred"].sort_index(),
    "GBM features_all": experiments[("features_all", "temporal")]["pred"].sort_index(),
    "naive_estacional_lag_12": df_features.loc[test_temporal.index, "lag_12"],
})

fig, ax = plt.subplots(figsize=(12, 5))
temporal_plot.plot(ax=ax, marker="o")
ax.set_title("Predicciones en holdout temporal")
ax.set_ylabel("Pasajeros")
ax.legend()
plt.show()

display(temporal_plot.head())

## Parte 6) Interpretabilidad, 20 puntos

La interpretabilidad se calcula sobre el modelo entrenado con `features_all` y holdout temporal, que es la combinacion valida para simular prediccion futura. Se grafica el top 10 de importancia nativa y el top 10 de permutation importance sobre test temporal con `scoring="neg_mean_absolute_error"` y `n_repeats=30`.


In [ ]:
valid_result = experiments[("features_all", "temporal")]
valid_model = valid_result["model"]
X_test_valid = valid_result["X_test"]
y_test_valid = valid_result["y_test"]

native_importance = (
    pd.DataFrame({
        "feature": features_all,
        "importance": valid_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
native_importance.head(10).sort_values("importance").plot.barh(
    x="feature", y="importance", ax=ax, legend=False
)
ax.set_title("Top 10 - Importancia nativa del GBM")
ax.set_xlabel("Importancia")
plt.show()

display(native_importance.head(10))

In [ ]:
perm = permutation_importance(
    valid_model,
    X_test_valid,
    y_test_valid,
    scoring="neg_mean_absolute_error",
    n_repeats=30,
    random_state=RANDOM_STATE,
)

permutation_df = (
    pd.DataFrame({
        "feature": features_all,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
permutation_df.head(10).sort_values("importance_mean").plot.barh(
    x="feature", y="importance_mean", ax=ax, legend=False, xerr="importance_std"
)
ax.set_title("Top 10 - Permutation importance en test temporal")
ax.set_xlabel("Aumento esperado del error MAE al permutar")
plt.show()

display(permutation_df.head(10))

## Parte 7) Preguntas de analisis, 20 puntos

La siguiente celda arma una lectura cuantitativa a partir de los resultados calculados arriba. Luego se responden las cuatro preguntas conceptuales solicitadas.


In [ ]:
temporal_results = results[results["holdout"] == "temporal"].set_index("features")
shuffle_results = results[results["holdout"] == "shuffle"].set_index("features")

mae_fecha = temporal_results.loc["features_fecha", "MAE"]
mae_all = temporal_results.loc["features_all", "MAE"]
mase_all = temporal_results.loc["features_all", "MASE"]
mejora_mae = (mae_fecha - mae_all) / mae_fecha * 100

top_native = native_importance.iloc[0]["feature"]
top_perm = permutation_df.iloc[0]["feature"]

print("Resumen para responder preguntas")
print("- Holdout temporal: representa predecir futuro usando pasado.")
print("- Holdout shuffle: mezcla fechas; sirve como contraste, no como validacion real de forecasting.")
print(f"- MAE temporal features_fecha: {mae_fecha:.3f}")
print(f"- MAE temporal features_all:   {mae_all:.3f}")
print(f"- Cambio relativo de MAE al agregar lags/rolling/expanding: {mejora_mae:.2f}%")
print(f"- MASE temporal features_all: {mase_all:.3f}")
print("  Interpretacion MASE: < 1 mejora al naive estacional; > 1 pierde contra ese baseline.")
print(f"- Feature top en importancia nativa: {top_native}")
print(f"- Feature top en permutation importance: {top_perm}")

### Pregunta 1: holdout temporal vs holdout con shuffle

El holdout temporal refleja mejor el escenario real de uso del modelo, porque en forecasting se entrena con pasado y se predice futuro. Este esquema respeta el "cuando": al momento de predecir los ultimos 24 meses, el modelo solo pudo haber observado informacion anterior.

El holdout con shuffle mezcla filas al azar. Eso puede hacer que fechas futuras queden en entrenamiento y fechas pasadas queden en test. Aunque no se use directamente la variable objetivo futura como feature, se rompe la estructura temporal del problema y se contamina la evaluacion: ya no se mide una prediccion futura realista, sino la capacidad de reconocer patrones globales de la serie con informacion temporal mezclada.

### Pregunta 2: features_fecha vs features_all en holdout temporal

`features_fecha` captura solamente el calendario: mes, trimestre, anio, dia del anio y codificaciones ciclicas. Estas variables representan el "cuando" y permiten aprender estacionalidad.

`features_all` agrega lags, rolling windows y expanding windows. Esas variables representan el "que venia pasando": nivel reciente, estacionalidad del mismo mes del anio anterior, tendencia local y variabilidad historica. El aporte concreto de estas variables se observa comparando las metricas del holdout temporal en la tabla de resultados y en el resumen cuantitativo anterior.

### Pregunta 3: interpretacion de MASE

MASE compara el error del modelo contra el error promedio de un naive estacional de lag 12. En esta serie, ese baseline es fuerte porque AirPassengers tiene una estacionalidad anual muy marcada.

- Si MASE < 1, el modelo mejora al naive estacional.
- Si MASE = 1, el modelo empata aproximadamente con el naive estacional.
- Si MASE > 1, el modelo pierde contra el naive estacional.

Si un GBM no mejora ampliamente a este baseline, el resultado sigue siendo razonable: la serie es corta, mensual, muy estacional y con tendencia creciente. Mejorar un naive estacional fuerte exige capturar nivel, tendencia y estacionalidad sin sobreajustar.

### Pregunta 4: feature dominante en los rankings

La feature dominante se interpreta comparando la importancia nativa con permutation importance. En esta tarea es esperable que aparezcan arriba variables ligadas al mismo mes del anio anterior, especialmente `lag_12`, o features de calendario/ciclo anual. Tiene sentido porque la serie AirPassengers presenta un patron anual muy marcado: los meses altos y bajos tienden a repetirse cada anio, aunque sobre un nivel creciente.


## Conclusiones

- El holdout temporal es la evaluacion valida para un problema de forecasting.
- El holdout con shuffle se incluye por consigna, pero no debe usarse para concluir desempeno real futuro.
- Las features rolling y expanding se construyen con `.shift(1)` para evitar leakage temporal.
- `features_all` permite incorporar informacion historica de la serie ademas del calendario.
- MASE permite comparar el modelo contra un baseline estacional fuerte de lag 12.
- La interpretabilidad permite verificar si el modelo usa seniales coherentes con la estructura estacional de AirPassengers.
